# 09A – Model Serialization & Production Packaging

Prepare the finalized bankruptcy prediction model for production deployment.

## Business Objective
Package the trained model with metadata, verify it can be reloaded, and create deployment-ready artifacts.

In [ ]:
import joblib
import json
import hashlib
from datetime import datetime
from pathlib import Path
import pandas as pd

MODEL_PATH='production_bankruptcy_model.joblib'
DATA_PATH='american_bankruptcy_cleaned.csv'


In [ ]:
# Load model and dataset
model=joblib.load(MODEL_PATH)
df=pd.read_csv(DATA_PATH)

target='status_label' if 'status_label' in df.columns else 'target'
feature_names=[c for c in df.columns if c!=target]

print(type(model))
print(f'Features: {len(feature_names)}')


In [ ]:
# Verify model inference
sample=df.drop(columns=[target]).head(5)
pred=model.predict(sample)
prob=model.predict_proba(sample)

verification=pd.DataFrame({
    'Prediction':pred,
    'Probability':prob.max(axis=1)
})
verification.to_csv('model_verification_predictions.csv',index=False)
verification


In [ ]:
# Create metadata
sha256=hashlib.sha256(Path(MODEL_PATH).read_bytes()).hexdigest()

metadata={
    "model_name":"Bankruptcy Risk Prediction",
    "model_file":MODEL_PATH,
    "algorithm":type(model).__name__,
    "version":"1.0.0",
    "dataset":"american_bankruptcy_cleaned.csv",
    "feature_count":len(feature_names),
    "created_utc":datetime.utcnow().isoformat()+"Z",
    "sha256":sha256
}

with open("model_metadata.json","w") as f:
    json.dump(metadata,f,indent=4)

pd.DataFrame({
    "Feature":feature_names
}).to_csv("model_schema.csv",index=False)

metadata


## Production Checklist

- ✔ Model successfully loads
- ✔ Prediction verified
- ✔ Feature schema exported
- ✔ Model metadata generated
- ✔ Ready for API, Streamlit, and Docker deployment

## Deliverables

- `model_metadata.json`
- `model_schema.csv`
- `model_verification_predictions.csv`

These artifacts are used by the remaining deployment notebooks.